# Scenario 16: Quote-To-Cash (Group Chat)

| Field | Value |
| --- | --- |
| Scenario id | `scenario-16-quote-to-cash-group-chat` |
| Pattern | `group-chat` |
| API | `Invocations API` |

**Learning goal:** Learn collaborative quote review where reviewers debate CRM readiness, customer fit, SKU selection, validity, and pricing risk, and the quote owner closes each round with a readiness verdict.

> Use Group Chat when a visible debate and cross-check between roles improves the quote before it is finalized.


In [ ]:
import os
import re as _re

from IPython.display import HTML, display


_AGENT_COLORS = ("#3868c8", "#b0530f", "#2f7d4f", "#7d3f98", "#a3374b", "#0f7d8a", "#8a6d0f", "#54596b")

_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
.maf-callout {
    border-left: 4px solid #3868c8; border-radius: 6px; padding: 0.6em 0.9em;
    margin: 0.6em 0; background: rgba(56, 104, 200, 0.08);
}
.maf-roster { display: flex; flex-wrap: wrap; gap: 0.6em; margin: 0.4em 0; }
.maf-card {
    border: 1px solid rgba(128, 128, 128, 0.35); border-radius: 8px;
    padding: 0.55em 0.8em; min-width: 14em; max-width: 24em; flex: 1;
}
.maf-card b { display: block; margin-bottom: 0.15em; }
.maf-card small { opacity: 0.75; }
.maf-chip {
    display: inline-block; border-radius: 999px; padding: 0 0.6em; margin: 0.2em 0.2em 0 0;
    font-size: 0.78em; border: 1px solid rgba(128, 128, 128, 0.4);
}
.maf-turn {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.45em 0.8em; margin: 0.45em 0; background: rgba(128, 128, 128, 0.07);
    white-space: pre-wrap;
}
.maf-turn b { color: var(--maf-agent, inherit); }
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


def _escape_html(value) -> str:
    import html as _html

    return _html.escape(str(value))


def agent_color(name: str) -> str:
    """Deterministic per-agent accent color, stable across cells and runs."""

    return _AGENT_COLORS[sum(ord(ch) for ch in name) % len(_AGENT_COLORS)]


def render_callout(text: str) -> None:
    display(HTML("<div class='maf-callout'>" + _escape_html(text) + "</div>"))


def render_roster(scenario) -> None:
    """Render the agent roster as color-accented cards with tool chips."""

    cards = []
    for spec in scenario.agents:
        chips = "".join(
            "<span class='maf-chip'>" + _escape_html(tool) + "</span>" for tool in spec.mcp_tools
        ) or "<span class='maf-chip'>instructions only</span>"
        cards.append(
            "<div class='maf-card' style='border-top: 3px solid " + agent_color(spec.name) + "'>"
            + "<b>" + _escape_html(spec.name) + "</b>"
            + "<small>" + _escape_html(spec.description) + "</small>"
            + "<div>" + chips + "</div></div>"
        )
    display(HTML("<div class='maf-roster'>" + "".join(cards) + "</div>"))


_TURN_LABEL = _re.compile(r"^\[([A-Za-z0-9_]+)\]\s*", _re.MULTILINE)


def render_transcript(text: str) -> None:
    """Render workflow output as color-coded per-agent turns; plain print fallback."""

    pieces = _TURN_LABEL.split(text)
    turns = []
    preamble = pieces[0].strip()
    if preamble:
        turns.append("<div class='maf-turn'>" + _escape_html(preamble) + "</div>")
    for label, body in zip(pieces[1::2], pieces[2::2]):
        turns.append(
            "<div class='maf-turn' style='--maf-agent: " + agent_color(label) + "'>"
            + "<b>" + _escape_html(label) + "</b><br>" + _escape_html(body.strip()) + "</div>"
        )
    if turns:
        display(HTML("<div>" + "".join(turns) + "</div>"))
    else:
        print(text)


OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:14b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

apply_notebook_style()
print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


## Pattern: Group Chat

Group chat orchestration creates a visible multi-agent discussion. A selector function chooses the next participant round-robin, and a per-scenario termination function checks the closing message of each full cycle, so the synthesizer or chair always speaks last.

Best fit: decisions that benefit from critique, tradeoffs, and a short transcript.

## API Shape

**Invocations API -- per-request job payload shape.** Each request body carries `scenario`, `pattern`, `task`, `artifacts`, and `constraints`. The caller controls which orchestration runs per invocation. This fits webhooks, CI pipelines, schedulers, and service-to-service calls where the task definition changes with every request.

This is a Scenario 16 quote-to-cash variant. The same six business roles (CRM trigger, customer context, SKU discovery, product fit, pricing and legal, quote generation) appear in every Scenario 16 notebook -- only the orchestration pattern changes. Compare notebooks 16a-16e to see how the same roles behave under sequential, concurrent, handoff, group-chat, and magentic coordination.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Agents take turns in cycles; the last agent closes each cycle and can end the chat. |
| Coordination | A selector function rotates speakers; termination is checked only at cycle boundaries. |
| Output | The transcript shows critique, refinement, and a closing verdict. |
| Best when | Use when visible debate improves the answer. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `QuoteTriggerAgent` | Argues from CRM readiness. | Domain tools: `crm_get_quote_trigger` |
| `CustomerContextAgent` | Argues from the customer's context. | Domain tools: `crm_get_customer_profile` |
| `SkuDiscoveryAgent` | Proposes and defends the SKU selection. | Domain tools: `product_search_catalog` |
| `ProductFitAgent` | Challenges SKU validity and compatibility. | Domain tools: `product_validate_skus` |
| `PricingTermsAgent` | Challenges pricing and legal risk. | Domain tools: `pricing_calculate_quote`, `legal_evaluate_terms` |
| `QuoteGenerationAgent` | Synthesizes the debate and closes each round. | Domain tools: `crm_get_quote_trigger`, `crm_get_customer_profile`, `product_search_catalog`, `product_validate_skus`, `pricing_calculate_quote`, `legal_evaluate_terms`, `quote_format_package` |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


## MCP Tool Context

In production, these quote-to-cash context functions are exposed by a local FastMCP stdio server and attached to
instruction-led LLM agents with `MCPStdioTool` using per-agent allowed tools. This notebook inlines
the same domain functions as plain callable tools so it remains standalone.


In [ ]:
from typing import Any


_QUOTE_TRIGGERS: dict[str, dict[str, Any]] = {
    "OPP-5001": {
        "opportunity_id": "OPP-5001",
        "account_id": "ACC-3300",
        "stage": "Negotiation",
        "quote_ready": True,
        "trigger_conditions": [
            "Opportunity stage is Negotiation or later.",
            "Primary contact and billing account are set.",
            "Budget is confirmed by the customer.",
        ],
        "blocking_conditions": [],
    },
    "OPP-5002": {
        "opportunity_id": "OPP-5002",
        "account_id": "ACC-3301",
        "stage": "Discovery",
        "quote_ready": False,
        "trigger_conditions": ["Opportunity stage is Negotiation or later."],
        "blocking_conditions": [
            "Opportunity is still in Discovery.",
            "No confirmed budget on the opportunity.",
        ],
    },
}

_CUSTOMER_PROFILES: dict[str, dict[str, Any]] = {
    "ACC-3300": {
        "account_id": "ACC-3300",
        "customer_name": "Contoso Manufacturing",
        "address": "120 Industrial Way, Aurora, IL 60502, USA",
        "msa_status": "signed",
        "account_status": "active",
        "segment": "enterprise",
        "buying_context": "Expanding plant automation; standardizing on one analytics platform.",
    },
    "ACC-3301": {
        "account_id": "ACC-3301",
        "customer_name": "Fabrikam Logistics",
        "address": "44 Harbor Rd, Tacoma, WA 98402, USA",
        "msa_status": "in_review",
        "account_status": "active",
        "segment": "mid-market",
        "buying_context": "Evaluating route-optimization add-ons for peak season.",
    },
}

_CATALOG: tuple[dict[str, Any], ...] = (
    {"sku": "SKU-ANALYTICS-CORE", "name": "Analytics Core Platform", "bundle": "platform", "list_price": 48000, "keywords": ("analytics", "platform", "core")},
    {"sku": "SKU-ANALYTICS-EDGE", "name": "Edge Connector Pack", "bundle": "platform", "list_price": 12000, "keywords": ("analytics", "edge", "connector", "automation")},
    {"sku": "SKU-SUPPORT-PREM", "name": "Premier Support (12 mo)", "bundle": "support", "list_price": 9000, "keywords": ("support", "premier", "service")},
    {"sku": "SKU-ROUTE-OPT", "name": "Route Optimization Add-on", "bundle": "logistics", "list_price": 15000, "keywords": ("route", "optimization", "logistics")},
    {"sku": "SKU-TRAINING-1", "name": "Onboarding & Training", "bundle": "services", "list_price": 6000, "keywords": ("training", "onboarding", "services")},
)

_SKU_INDEX = {entry["sku"]: entry for entry in _CATALOG}
_INCOMPATIBLE_PAIRS = {("SKU-ROUTE-OPT", "SKU-ANALYTICS-EDGE")}
_UNAVAILABLE_SKUS = {"SKU-TRAINING-1"}

_LEGAL_TERMS: dict[str, dict[str, Any]] = {
    "enterprise": {
        "segment": "enterprise",
        "risk_level": "medium",
        "clauses": [
            "Net-45 payment terms.",
            "Standard MSA governs; no bespoke indemnity without legal review.",
            "Auto-renewal with 60-day opt-out.",
        ],
        "approvals_required": ["Deal desk", "Legal (if discount > 20%)"],
    },
    "mid-market": {
        "segment": "mid-market",
        "risk_level": "low",
        "clauses": ["Net-30 payment terms.", "Click-through terms acceptable below $50k."],
        "approvals_required": ["Deal desk"],
    },
}



# Fixture data only -- the tools in the next cell read from these embedded records.
print("opportunities:", ", ".join(sorted(_QUOTE_TRIGGERS)))
print("accounts:     ", ", ".join(sorted(_CUSTOMER_PROFILES)))
print("catalog SKUs: ", ", ".join(entry["sku"] for entry in _CATALOG))


In [ ]:
import hashlib
import json
from typing import Any


def _string_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        pieces = value.replace(";", ",").replace("\n", ",").split(",")
        return [piece.strip() for piece in pieces if piece.strip()]
    try:
        items = iter(value)
    except TypeError:
        text = str(value).strip()
        return [text] if text else []
    flattened: list[str] = []
    for item in items:
        flattened.extend(_string_list(item))
    return flattened


def crm_get_quote_trigger(opportunity_id: str = "OPP-5001") -> dict[str, Any]:
    """Return CRM trigger state for an opportunity."""

    key = (opportunity_id or "").strip().upper()
    record = _QUOTE_TRIGGERS.get(key)
    if record is None:
        return {"found": False, "opportunity_id": opportunity_id, "known_ids": sorted(_QUOTE_TRIGGERS)}
    return {"found": True, **record}


def crm_get_customer_profile(account_id: str = "ACC-3300") -> dict[str, Any]:
    """Return the enriched CRM customer profile for an account."""

    key = (account_id or "").strip().upper()
    record = _CUSTOMER_PROFILES.get(key)
    if record is None:
        return {"found": False, "account_id": account_id, "known_ids": sorted(_CUSTOMER_PROFILES)}
    return {"found": True, **record}


def product_search_catalog(query: str = "analytics platform") -> dict[str, Any]:
    """Search the product/SKU catalog with a simple keyword match."""

    terms = [term for term in (query or "").lower().replace(",", " ").split() if term]
    scored: list[tuple[int, dict[str, Any]]] = []
    for entry in _CATALOG:
        haystack = " ".join((entry["name"], entry["bundle"], " ".join(entry["keywords"]))).lower()
        score = sum(1 for term in terms if term in haystack)
        if score or not terms:
            scored.append((score, entry))
    scored.sort(key=lambda item: (-item[0], item[1]["sku"]))
    matches = [
        {"sku": e["sku"], "name": e["name"], "bundle": e["bundle"], "list_price": e["list_price"], "match_score": s}
        for s, e in scored
    ]
    return {"query": query, "match_count": len(matches), "matches": matches}


def product_validate_skus(skus: str = "") -> dict[str, Any]:
    """Validate SKU compatibility, availability, and completeness."""

    requested = _string_list(skus) or [entry["sku"] for entry in _CATALOG[:2]]
    requested_set = {sku.strip().upper() for sku in requested}
    validated: list[dict[str, Any]] = []
    for sku in requested:
        key = sku.strip().upper()
        known = key in _SKU_INDEX
        available = known and key not in _UNAVAILABLE_SKUS
        compatible = not any(
            {key, other} == set(pair) for pair in _INCOMPATIBLE_PAIRS for other in requested_set
        )
        validated.append(
            {
                "sku": key,
                "known": known,
                "compatible": compatible,
                "available": available,
                "complete": bool(known and available and compatible),
            }
        )
    all_valid = bool(validated) and all(item["complete"] for item in validated)
    return {"requested": requested, "validated": validated, "all_valid": all_valid}


def pricing_calculate_quote(skus: str = "", discount_pct: float = 0.0) -> dict[str, Any]:
    """Calculate quote pricing for a set of SKUs."""

    requested = _string_list(skus) or [entry["sku"] for entry in _CATALOG[:2]]
    line_items: list[dict[str, Any]] = []
    subtotal = 0
    for sku in requested:
        key = sku.strip().upper()
        entry = _SKU_INDEX.get(key)
        price = int(entry["list_price"]) if entry else 0
        subtotal += price
        line_items.append({"sku": key, "list_price": price, "in_catalog": entry is not None})
    try:
        pct = float(discount_pct)
    except (TypeError, ValueError):
        pct = 0.0
    pct = max(0.0, min(40.0, pct))
    discount = round(subtotal * pct / 100.0, 2)
    total = round(subtotal - discount, 2)
    return {
        "currency": "USD",
        "line_items": line_items,
        "subtotal": subtotal,
        "discount_pct": pct,
        "discount": discount,
        "total": total,
    }


def legal_evaluate_terms(segment: str = "enterprise", discount_pct: float = 0.0) -> dict[str, Any]:
    """Return legal/contract terms and required approvals for a segment."""

    key = (segment or "").strip().lower()
    terms = _LEGAL_TERMS.get(key, _LEGAL_TERMS["enterprise"])
    try:
        pct = float(discount_pct)
    except (TypeError, ValueError):
        pct = 0.0
    approvals = list(terms["approvals_required"])
    if pct > 20 and "Legal review" not in approvals:
        approvals.append("Legal review (discount over 20%)")
    return {
        "segment": terms["segment"],
        "risk_level": terms["risk_level"],
        "clauses": list(terms["clauses"]),
        "approvals_required": approvals,
    }


def quote_format_package(
    customer_name: str = "Contoso Manufacturing",
    total: float = 0.0,
    skus: str = "",
) -> dict[str, Any]:
    """Format the final customer-ready quote package."""

    requested = _string_list(skus)
    fingerprint = "|".join([customer_name, ",".join(requested), f"{float(total or 0.0):.2f}"])
    digest = hashlib.sha256(fingerprint.encode("utf-8")).hexdigest()[:8]
    return {
        "quote_id": f"Q2C-{digest}",
        "format": "pdf",
        "customer_name": customer_name,
        "total": round(float(total or 0.0), 2),
        "skus": [sku.strip().upper() for sku in requested],
        "sections": ["Cover", "Customer & MSA", "Line Items & Pricing", "Terms & Conditions", "Signature"],
        "customer_ready": True,
    }


MCP_TOOL_FUNCTIONS.update(
    {
        "crm_get_quote_trigger": crm_get_quote_trigger,
        "crm_get_customer_profile": crm_get_customer_profile,
        "product_search_catalog": product_search_catalog,
        "product_validate_skus": product_validate_skus,
        "pricing_calculate_quote": pricing_calculate_quote,
        "legal_evaluate_terms": legal_evaluate_terms,
        "quote_format_package": quote_format_package,
    }
)

# Demo (offline): call one grounded tool directly before any agent exists.
print(json.dumps(crm_get_quote_trigger("OPP-5001"), indent=2))


In [ ]:
import json
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_task: str
    agents: tuple[AgentSpec, ...]
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None
    termination_phrases: tuple[str, ...] = ()


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "scenario-16-quote-to-cash-group-chat",
  "pattern": "group-chat",
  "title": "Scenario 16: Quote-To-Cash (Group Chat)",
  "learning_goal": "Learn collaborative quote review where reviewers debate CRM readiness, customer fit, SKU selection, validity, and pricing risk, and the quote owner closes each round with a readiness verdict.",
  "when_to_use": "Use Group Chat when a visible debate and cross-check between roles improves the quote before it is finalized.",
  "sample_task": "Create a quote for opportunity OPP-5001 (account ACC-3300, Contoso Manufacturing). They want the analytics platform with edge connectors and premier support, with standard enterprise terms, and they are targeting a 25 percent discount -- check what approvals that requires.",
  "handoff_finisher": null,
  "concurrent_synthesizer": null,
  "termination_phrases": [
    "final quote recommendation"
  ],
  "agents": [
    {
      "name": "QuoteTriggerAgent",
      "description": "Argues from CRM readiness.",
      "instructions": "Argue whether the CRM conditions justify quoting now. Use crm_get_quote_trigger for the trigger facts and challenge the council if blockers exist. Do not invent CRM data.",
      "mcp_tools": [
        "crm_get_quote_trigger"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    },
    {
      "name": "CustomerContextAgent",
      "description": "Argues from the customer's context.",
      "instructions": "Argue whether the proposed quote fits this customer. Use crm_get_customer_profile and challenge SKU or terms choices that conflict with the account's segment, MSA status, or buying context.",
      "mcp_tools": [
        "crm_get_customer_profile"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    },
    {
      "name": "SkuDiscoveryAgent",
      "description": "Proposes and defends the SKU selection.",
      "instructions": "Propose the SKU set that best fits the request. Use product_search_catalog, state list prices, and defend or revise your proposal when other reviewers challenge it.",
      "mcp_tools": [
        "product_search_catalog"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    },
    {
      "name": "ProductFitAgent",
      "description": "Challenges SKU validity and compatibility.",
      "instructions": "Challenge the proposed SKUs. Use product_validate_skus (comma-separated SKU strings) and call out unknown, unavailable, or incompatible SKUs before the council converges.",
      "mcp_tools": [
        "product_validate_skus"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    },
    {
      "name": "PricingTermsAgent",
      "description": "Challenges pricing and legal risk.",
      "instructions": "Challenge the pricing and terms. Use pricing_calculate_quote (comma-separated SKU strings) and legal_evaluate_terms, and flag discounts or clauses that need approvals.",
      "mcp_tools": [
        "pricing_calculate_quote",
        "legal_evaluate_terms"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    },
    {
      "name": "QuoteGenerationAgent",
      "description": "Synthesizes the debate and closes each round.",
      "instructions": "Synthesize the round: state the current SKU set, pricing, terms, and open challenges. Resolve gaps with your tools and quote_format_package when the package is ready. When the council has converged, end your turn with a line 'FINAL QUOTE RECOMMENDATION: <ready or not ready> - <one-sentence rationale>'.",
      "mcp_tools": [
        "crm_get_quote_trigger",
        "crm_get_customer_profile",
        "product_search_catalog",
        "product_validate_skus",
        "pricing_calculate_quote",
        "legal_evaluate_terms",
        "quote_format_package"
      ],
      "mcp_server": "quote_to_cash_context",
      "route_keywords": []
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_task=SCENARIO_DATA["sample_task"],
    agents=AGENTS,
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
    termination_phrases=tuple(SCENARIO_DATA.get("termination_phrases", [])),
)


def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "sample": getattr(scenario, "sample_task"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }
def build_invocation_prompt(payload: dict[str, object]) -> str:
    artifacts = "\n".join(f"- {item}" for item in payload.get("artifacts", [])) or "- No artifacts supplied."
    constraints = "\n".join(f"- {item}" for item in payload.get("constraints", [])) or "- No explicit constraints."
    return (
        f"Scenario: {payload['scenario']} - {SCENARIO.title}\n"
        f"Pattern: {payload['pattern']}\n"
        f"Learning goal: {SCENARIO.learning_goal}\n"
        f"Subject: {payload['subject']}\n"
        f"Task: {payload['task']}\n\n"
        f"Artifacts:\n{artifacts}\n\n"
        f"Constraints:\n{constraints}\n\n"
        "Session context:\nNo prior turns for this session.\n\n"
        "Return actionable findings. Do not claim to have inspected artifacts beyond the provided names and context."
    )

MAX_TOKENS = 500 if SCENARIO.pattern == "magentic" else 250
INVOCATION_PAYLOAD = {
    "scenario": SCENARIO.id,
    "pattern": SCENARIO.pattern,
    "task": SCENARIO.sample_task,
    "subject": "notebook sample",
    "artifacts": [],
    "constraints": [],
    "stream": False,
}
SAMPLE_PROMPT = build_invocation_prompt(INVOCATION_PAYLOAD)

render_roster(SCENARIO)
print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "qwen3:14b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "500")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


class ScenarioOllamaChatClient(OllamaChatClient):
    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


print("Agent factory ready: make_agent(spec) creates an instruction-led Ollama agent "
      "with its granted tools attached.")


In [ ]:
import re
from typing import Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))



print("Workflow plumbing ready: dispatch executor, shared transcript state, and "
      "request/response helpers.")


In [ ]:
def make_group_chat_termination(phrases: tuple[str, ...], participant_count: int, max_cycles: int = 2) -> Any:
    def should_stop(messages: list[Any]) -> bool:
        assistant = [m for m in messages if getattr(m, "role", None) == "assistant"]
        if not assistant or len(assistant) % participant_count != 0:
            return False
        if len(assistant) >= max_cycles * participant_count:
            return True
        last_text = (getattr(assistant[-1], "text", "") or "").lower()
        return bool(phrases) and all(phrase in last_text for phrase in phrases)

    return should_stop



# Demo (offline): termination only fires when the closing agent ends a full cycle.
class _DemoMsg:
    def __init__(self, text: str) -> None:
        self.role = "assistant"
        self.text = text


_n = len(SCENARIO.agents)
_phrase = " ".join(SCENARIO.termination_phrases) or "final recommendation"
_stop = make_group_chat_termination(SCENARIO.termination_phrases, _n)
print("mid-cycle, phrase present  ->", _stop([_DemoMsg(_phrase)] * max(1, _n - 1)))
print("cycle end, no phrase       ->", _stop([_DemoMsg("still debating")] * _n))
print("cycle end, phrase present  ->", _stop([_DemoMsg("x")] * (_n - 1) + [_DemoMsg(_phrase)]))
print("after two full cycles      ->", _stop([_DemoMsg("x")] * (2 * _n)))


In [ ]:
def build_group_chat_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import GroupChatBuilder

    participants = _agents_for(scenario, config=config)

    def round_robin_selector(state: Any) -> str:
        participant_names = list(state.participants.keys())
        return participant_names[state.current_round % len(participant_names)]

    return GroupChatBuilder(
        participants=participants,
        selection_func=round_robin_selector,
        termination_condition=make_group_chat_termination(
            scenario.termination_phrases, len(scenario.agents)
        ),
        intermediate_output_from=participants,
    ).build()



def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    return build_group_chat_workflow(scenario, config=config)


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
print(
    f"Built the {SCENARIO.pattern} workflow over {len(SCENARIO.agents)} agents: "
    + type(workflow).__name__
)


## Flow Diagram

The diagram below shows 6 participants in a round-robin loop with a code-defined termination function attached to the Invocations API /invocations.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _group_chat_diagram(scenario, api_boundary="Invocations API /invocations", input_label="Job payload")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram



def _group_chat_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> selector{Round-robin selector}")
    previous = "selector"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} --> {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> stop{{Termination condition}}")
    lines.append("    stop -->|continue| selector")
    lines.append("    stop -->|done| output[Invocation summary]")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)



def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Invocations API /invocations")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
quote_to_cash_diagram = display_quote_to_cash_flow(SCENARIO)
print(flow_diagram.mermaid)


In [ ]:
from collections.abc import Mapping, Sequence


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")



def group_chat_learning_summary(
    scenario: ScenarioSpec,
    prompt: str,
    framework_text: str,
) -> str:
    """Explain a completed group-chat run when this framework build hides the transcript."""

    lines = [
        "Group chat completed.",
        "",
        f"Framework result: {framework_text.strip()}",
        "",
        "Learning view:",
        "- The workflow used Microsoft Agent Framework's GroupChatBuilder with LLM-backed participants.",
        "- Selection is code-defined round robin; termination is code-defined from assistant messages.",
        f"- The submitted scenario prompt was: {prompt}",
        "- Participant order:",
    ]
    for index, spec in enumerate(scenario.agents, start=1):
        tools = ", ".join(spec.mcp_tools) if spec.mcp_tools else "no domain tools"
        lines.append(f"  {index}. {spec.name}: {spec.description} ({tools})")
    tool_names = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    if tool_names:
        lines.append("- Grounding sources available to tool-enabled agents:")
        for tool_name in tool_names:
            lines.append(f"  - {tool_name}")
    lines.extend(
        [
            "",
            "Note: this local Agent Framework build returned the group-chat termination marker",
            "without exposing participant turns through get_intermediate_outputs(). The notebook",
            "keeps the framework run intact and prints this learning summary so the scenario",
            "still explains the orchestration shape and agent responsibilities.",
        ]
    )
    return "\n".join(lines)


print("Result formatting ready: workflow_result_to_text(...) turns framework events "
      "into readable text.")


## Live Run

Participants speak in round-robin order, and termination is only checked when the last agent closes a full cycle -- so the synthesizer always gets the final word. The chat ends early when the scenario's termination phrases appear in that closing message, and unconditionally after two full cycles. Intermediate outputs from each participant are surfaced alongside the final transcript.

> **Instructor note:** `qwen3:14b` runs with `think: False` by default (extended reasoning off).
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

render_transcript(output_text)


## What to Inspect

Read the transcript chronologically. Later turns should respond to earlier critiques rather than restarting the discussion. Termination is checked only at the end of each full cycle: the chat stops early when the scenario's termination phrases appear in the closing agent's message, or after two full cycles -- check which condition fired and why.


## Experiments

- Change `INVOCATION_PAYLOAD['task']`, `subject`, `artifacts`, or `constraints` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster runs or raise it when group-chat needs more room.
